# Feature Engineering with DuckDB and SQL

This notebook creates the clinical feature table for the ICU mortality prediction project.

## Objectives

1. Connect Python to DuckDB.
2. Register MIMIC-IV CSV files as SQL views.
3. Load the base ICU cohort created in Notebook 2.
4. Create admission-time features.
5. Extract vital signs measured during the first 24 hours after ICU admission.
6. Extract laboratory measurements during the same 24-hour window.
7. Merge all features into one modeling dataset.
8. Validate and save the result.

## Prediction window

For every ICU stay:

```text
feature_start = intime
feature_end   = prediction_time = intime + 24 hours
```

Only measurements inside this interval are used.

## Important leakage rule

Variables known only after the prediction time, including ICU discharge time, hospital discharge information, and length of stay, are not used as predictors.

## 1. Imports and project paths

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data" / "raw" / "mimic-iv-clinical-database-demo-2.2"
HOSP_DIR = DATA_DIR / "hosp"
ICU_DIR = DATA_DIR / "icu"

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results" / "tables"

BASE_COHORT_PATH = PROCESSED_DIR / "icu_mortality_cohort_demo.csv"
OUTPUT_PATH = PROCESSED_DIR / "modeling_cohort_with_clinical_features.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("MIMIC directory:", DATA_DIR.resolve())
print("Base cohort:", BASE_COHORT_PATH.resolve())

Project directory: C:\Projects\clinical-outcome-prediction
MIMIC directory: C:\Projects\clinical-outcome-prediction\data\raw\mimic-iv-clinical-database-demo-2.2
Base cohort: C:\Projects\clinical-outcome-prediction\data\processed\icu_mortality_cohort_demo.csv


## 2. Verify required input files

In [2]:
required_files = {
    "base_cohort": BASE_COHORT_PATH,
    "patients": HOSP_DIR / "patients.csv.gz",
    "admissions": HOSP_DIR / "admissions.csv.gz",
    "icustays": ICU_DIR / "icustays.csv.gz",
    "chartevents": ICU_DIR / "chartevents.csv.gz",
    "d_items": ICU_DIR / "d_items.csv.gz",
    "labevents": HOSP_DIR / "labevents.csv.gz",
    "d_labitems": HOSP_DIR / "d_labitems.csv.gz",
}

missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

print("All required files were found.")

All required files were found.


## 3. Connect to DuckDB

In [3]:
con = duckdb.connect()

print("DuckDB connection created.")
print("DuckDB version:", duckdb.__version__)

DuckDB connection created.
DuckDB version: 1.5.5


## 4. Register MIMIC-IV files as SQL views

DuckDB can query compressed CSV files directly. No separate database server is required.

In [4]:
def sql_path(path: Path) -> str:
    """Return a DuckDB-safe path string."""
    return path.resolve().as_posix()


con.execute(
    f"""
    CREATE OR REPLACE VIEW patients AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["patients"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW admissions AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["admissions"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW icustays AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["icustays"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW chartevents AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["chartevents"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW d_items AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["d_items"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW labevents AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["labevents"])}',
        header = TRUE
    );
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW d_labitems AS
    SELECT *
    FROM read_csv_auto(
        '{sql_path(required_files["d_labitems"])}',
        header = TRUE
    );
    """
)

print("MIMIC views registered successfully.")

MIMIC views registered successfully.


## 5. Confirm that the SQL views work

In [5]:
table_counts = con.execute(
    """
    SELECT 'patients' AS table_name, COUNT(*) AS row_count FROM patients
    UNION ALL
    SELECT 'admissions', COUNT(*) FROM admissions
    UNION ALL
    SELECT 'icustays', COUNT(*) FROM icustays
    UNION ALL
    SELECT 'chartevents', COUNT(*) FROM chartevents
    UNION ALL
    SELECT 'd_items', COUNT(*) FROM d_items
    UNION ALL
    SELECT 'labevents', COUNT(*) FROM labevents
    UNION ALL
    SELECT 'd_labitems', COUNT(*) FROM d_labitems
    ORDER BY table_name;
    """
).df()

table_counts

,table_name,row_count
0,admissions,275
1,chartevents,668862
2,d_items,4014
3,d_labitems,1622
4,icustays,140
5,labevents,107727
6,patients,100


## 6. Load and register the base cohort

In [6]:
base_cohort = pd.read_csv(
    BASE_COHORT_PATH,
    parse_dates=["intime", "prediction_time", "outtime"],
)

required_cohort_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "intime",
    "prediction_time",
    "outtime",
    "gender",
    "anchor_age",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "first_careunit",
    "hospital_expire_flag",
]

missing_cohort_columns = [
    column
    for column in required_cohort_columns
    if column not in base_cohort.columns
]

if missing_cohort_columns:
    raise ValueError(
        f"Base cohort is missing columns: {missing_cohort_columns}"
    )

assert base_cohort["stay_id"].is_unique
assert base_cohort["hadm_id"].is_unique
assert base_cohort["intime"].notna().all()
assert base_cohort["prediction_time"].notna().all()
assert base_cohort["hospital_expire_flag"].isin([0, 1]).all()

con.register("base_cohort_df", base_cohort)

con.execute(
    """
    CREATE OR REPLACE TEMP VIEW base_cohort AS
    SELECT *
    FROM base_cohort_df;
    """
)

print("Base cohort shape:", base_cohort.shape)
base_cohort.head()

Base cohort shape: (128, 15)


,subject_id,hadm_id,stay_id,intime,prediction_time,outtime,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag
0,10023771,20044587,33177122,2113-08-25 09:32:41,2113-08-26 09:32:41,2113-08-27 16:27:53,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
1,10005909,20199380,36496303,2144-10-29 23:09:03,2144-10-30 23:09:03,2144-11-02 15:24:29,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
2,10003400,20214994,32128372,2137-02-25 23:37:19,2137-02-26 23:37:19,2137-03-10 21:29:36,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0
3,10008454,20291550,31959184,2110-11-30 17:11:36,2110-12-01 17:11:36,2110-12-05 16:48:24,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU),0
4,10019385,20297618,39268883,2180-02-21 08:34:06,2180-02-22 08:34:06,2180-02-22 16:05:14,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0


## 7. Validate the 24-hour prediction window

In [7]:
window_validation = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN prediction_time <= intime THEN 1
                ELSE 0
            END
        ) AS invalid_windows,
        MIN(prediction_time - intime) AS minimum_window,
        MAX(prediction_time - intime) AS maximum_window
    FROM base_cohort;
    """
).df()

window_validation

,total_rows,invalid_windows,minimum_window,maximum_window
0,128,0.0,1 days,1 days


In [8]:
assert window_validation.loc[0, "invalid_windows"] == 0

print("Prediction-window validation passed.")

Prediction-window validation passed.


## 8. Create admission-time features with SQL

These features are available at ICU admission and do not use future information.

In [9]:
admission_feature_sql = """
CREATE OR REPLACE TEMP VIEW admission_features AS

SELECT
    subject_id,
    hadm_id,
    stay_id,
    intime,
    prediction_time,
    outtime,
    gender,
    anchor_age,
    admission_type,
    admission_location,
    insurance,
    marital_status,
    race,
    first_careunit,
    hospital_expire_flag,

    EXTRACT('hour' FROM intime) AS icu_admission_hour,

    CASE
        WHEN EXTRACT('dow' FROM intime) IN (0, 6)
            THEN 1
        ELSE 0
    END AS weekend_admission,

    CASE
        WHEN UPPER(COALESCE(admission_type, ''))
             LIKE '%EMER%'
            THEN 1
        ELSE 0
    END AS emergency_admission,

    CASE
        WHEN UPPER(COALESCE(admission_location, ''))
             LIKE '%TRANSFER%'
            THEN 1
        ELSE 0
    END AS transfer_admission

FROM base_cohort;
"""

con.execute(admission_feature_sql)

admission_features = con.execute(
    """
    SELECT *
    FROM admission_features
    ORDER BY stay_id;
    """
).df()

print("Admission feature shape:", admission_features.shape)
admission_features.head()

Admission feature shape: (128, 19)


,subject_id,hadm_id,stay_id,intime,prediction_time,outtime,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag,icu_admission_hour,weekend_admission,emergency_admission,transfer_admission
0,10023117,28872262,30057454,2171-11-14 10:06:41,2171-11-15 10:06:41,2171-11-18 20:49:43,M,53,EW EMER.,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0,10,0,1,0
1,10032725,20611640,30101877,2143-03-22 06:42:00,2143-03-23 06:42:00,2143-03-25 15:05:33,F,38,EW EMER.,EMERGENCY ROOM,Other,SINGLE,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0,6,0,1,0
2,10016742,27568122,30425410,2178-07-22 08:19:00,2178-07-23 08:19:00,2178-07-25 16:42:43,F,58,OBSERVATION ADMIT,EMERGENCY ROOM,Medicaid,SINGLE,BLACK/AFRICAN AMERICAN,Medical Intensive Care Unit (MICU),0,8,0,0,0
3,10031757,28477280,30458995,2137-10-12 22:44:57,2137-10-13 22:44:57,2137-10-14 17:08:34,F,67,DIRECT EMER.,CLINIC REFERRAL,Other,DIVORCED,WHITE,Surgical Intensive Care Unit (SICU),0,22,1,1,0
4,10022281,29642388,30585761,2125-06-17 04:12:54,2125-06-18 04:12:54,2125-06-18 14:55:55,M,84,EW EMER.,EMERGENCY ROOM,Other,MARRIED,OTHER,Cardiac Vascular Intensive Care Unit (CVICU),0,4,1,1,0


## 9. Inspect candidate vital-sign labels

In [10]:
candidate_vital_items = con.execute(
    """
    SELECT
        itemid,
        label,
        abbreviation,
        category,
        unitname
    FROM d_items
    WHERE
        LOWER(label) LIKE '%heart rate%'
        OR LOWER(label) LIKE '%respiratory rate%'
        OR LOWER(label) LIKE '%oxygen saturation%'
        OR LOWER(label) LIKE '%spo2%'
        OR LOWER(label) LIKE '%temperature%'
        OR LOWER(label) LIKE '%blood pressure systolic%'
        OR LOWER(label) LIKE '%arterial blood pressure mean%'
        OR LOWER(label) LIKE '%non invasive blood pressure mean%'
    ORDER BY label, itemid;
    """
).df()

candidate_vital_items

,itemid,label,abbreviation,category,unitname
0,220052,Arterial Blood Pressure mean,ABPm,Routine Vital Signs,mmHg
1,220050,Arterial Blood Pressure systolic,ABPs,Routine Vital Signs,mmHg
2,226329,Blood Temperature CCO (C),Blood Temp CCO (C),Routine Vital Signs,°C
3,229236,Cerebral Temperature (C),Cerebral T (C),Hemodynamics,°C
4,224674,Changes in Temperature,Changes in Temperature,Toxicology,NaN
5,229862,Forehead SpO2 Sensor in Place,Forehead SpO2 Sensor in Place,Routine Vital Signs,NaN
6,220045,Heart Rate,HR,Routine Vital Signs,bpm
7,220047,Heart Rate Alarm - Low,HR Alarm - Low,Alarms,bpm
8,220046,Heart rate Alarm - High,HR Alarm - High,Alarms,bpm
9,224167,Manual Blood Pressure Systolic Left,Manual BPs L,Routine Vital Signs,mmHg


## 10. Map chart events to standardized vital-sign names

The mapping uses labels from `d_items` instead of relying on a single hard-coded `itemid`.

Temperature values documented in Fahrenheit are converted to Celsius.

In [11]:
vital_events_sql = """
CREATE OR REPLACE TEMP VIEW standardized_vital_events AS

SELECT
    c.stay_id,
    ce.charttime,

    CASE
        WHEN LOWER(di.label) = 'heart rate'
            THEN 'heart_rate'

        WHEN LOWER(di.label) IN (
            'respiratory rate',
            'respiratory rate (total)'
        )
            THEN 'respiratory_rate'

        WHEN LOWER(di.label) LIKE '%o2 saturation pulseoxymetry%'
             OR LOWER(di.label) LIKE '%oxygen saturation%'
             OR LOWER(di.label) = 'spo2'
            THEN 'spo2'

        WHEN LOWER(di.label) IN (
            'non invasive blood pressure systolic',
            'arterial blood pressure systolic',
            'nbp systolic',
            'abp systolic'
        )
            THEN 'sbp'

        WHEN LOWER(di.label) IN (
            'non invasive blood pressure mean',
            'arterial blood pressure mean',
            'nbp mean',
            'abp mean'
        )
            THEN 'map'

        WHEN LOWER(di.label) LIKE '%temperature%'
            THEN 'temperature_c'

        ELSE NULL
    END AS vital_name,

    CASE
        WHEN LOWER(di.label) LIKE '%temperature%'
             AND (
                 LOWER(COALESCE(di.unitname, '')) LIKE '%f%'
                 OR LOWER(di.label) LIKE '%fahrenheit%'
                 OR ce.valuenum > 70
             )
            THEN (ce.valuenum - 32.0) * 5.0 / 9.0

        ELSE ce.valuenum
    END AS standardized_value

FROM base_cohort AS c

INNER JOIN chartevents AS ce
    ON c.stay_id = ce.stay_id
    AND ce.charttime >= c.intime
    AND ce.charttime <= c.prediction_time

INNER JOIN d_items AS di
    ON ce.itemid = di.itemid

WHERE
    ce.valuenum IS NOT NULL
    AND (
        LOWER(di.label) = 'heart rate'
        OR LOWER(di.label) IN (
            'respiratory rate',
            'respiratory rate (total)'
        )
        OR LOWER(di.label) LIKE '%o2 saturation pulseoxymetry%'
        OR LOWER(di.label) LIKE '%oxygen saturation%'
        OR LOWER(di.label) = 'spo2'
        OR LOWER(di.label) IN (
            'non invasive blood pressure systolic',
            'arterial blood pressure systolic',
            'nbp systolic',
            'abp systolic'
        )
        OR LOWER(di.label) IN (
            'non invasive blood pressure mean',
            'arterial blood pressure mean',
            'nbp mean',
            'abp mean'
        )
        OR LOWER(di.label) LIKE '%temperature%'
    );
"""

con.execute(vital_events_sql)

vital_event_counts = con.execute(
    """
    SELECT
        vital_name,
        COUNT(*) AS measurement_count,
        COUNT(DISTINCT stay_id) AS stays_with_measurement
    FROM standardized_vital_events
    WHERE vital_name IS NOT NULL
    GROUP BY vital_name
    ORDER BY vital_name;
    """
).df()

vital_event_counts

,vital_name,measurement_count,stays_with_measurement
0,heart_rate,3494,128
1,map,3642,127
2,respiratory_rate,3764,128
3,sbp,3618,127
4,spo2,4151,128
5,temperature_c,1115,127


## 11. Apply broad clinical plausibility checks

Clearly impossible or likely unit-error values are changed to `NULL`.

These are quality-control boundaries, not diagnostic cutoffs.

In [12]:
con.execute(
    """
    CREATE OR REPLACE TEMP VIEW cleaned_vital_events AS

    SELECT
        stay_id,
        charttime,
        vital_name,

        CASE
            WHEN vital_name = 'heart_rate'
                 AND standardized_value BETWEEN 20 AND 300
                THEN standardized_value

            WHEN vital_name = 'respiratory_rate'
                 AND standardized_value BETWEEN 3 AND 80
                THEN standardized_value

            WHEN vital_name = 'spo2'
                 AND standardized_value BETWEEN 0 AND 100
                THEN standardized_value

            WHEN vital_name = 'sbp'
                 AND standardized_value BETWEEN 30 AND 300
                THEN standardized_value

            WHEN vital_name = 'map'
                 AND standardized_value BETWEEN 20 AND 250
                THEN standardized_value

            WHEN vital_name = 'temperature_c'
                 AND standardized_value BETWEEN 25 AND 45
                THEN standardized_value

            ELSE NULL
        END AS value

    FROM standardized_vital_events

    WHERE vital_name IS NOT NULL;
    """
)

vital_cleaning_summary = con.execute(
    """
    SELECT
        vital_name,
        COUNT(*) AS total_mapped_measurements,
        SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END)
            AS excluded_measurements,
        SUM(CASE WHEN value IS NOT NULL THEN 1 ELSE 0 END)
            AS retained_measurements
    FROM cleaned_vital_events
    GROUP BY vital_name
    ORDER BY vital_name;
    """
).df()

vital_cleaning_summary

,vital_name,total_mapped_measurements,excluded_measurements,retained_measurements
0,heart_rate,3494,0.0,3494.0
1,map,3642,10.0,3632.0
2,respiratory_rate,3764,14.0,3750.0
3,sbp,3618,0.0,3618.0
4,spo2,4151,0.0,4151.0
5,temperature_c,1115,0.0,1115.0


## 12. Aggregate vital signs by ICU stay

In [13]:
vital_feature_sql = """
CREATE OR REPLACE TEMP VIEW vital_features AS

SELECT
    stay_id,

    MIN(CASE WHEN vital_name = 'heart_rate' THEN value END)
        AS heart_rate_min,
    MAX(CASE WHEN vital_name = 'heart_rate' THEN value END)
        AS heart_rate_max,
    AVG(CASE WHEN vital_name = 'heart_rate' THEN value END)
        AS heart_rate_mean,

    MIN(CASE WHEN vital_name = 'respiratory_rate' THEN value END)
        AS respiratory_rate_min,
    MAX(CASE WHEN vital_name = 'respiratory_rate' THEN value END)
        AS respiratory_rate_max,
    AVG(CASE WHEN vital_name = 'respiratory_rate' THEN value END)
        AS respiratory_rate_mean,

    MIN(CASE WHEN vital_name = 'spo2' THEN value END)
        AS spo2_min,
    MAX(CASE WHEN vital_name = 'spo2' THEN value END)
        AS spo2_max,
    AVG(CASE WHEN vital_name = 'spo2' THEN value END)
        AS spo2_mean,

    MIN(CASE WHEN vital_name = 'sbp' THEN value END)
        AS sbp_min,
    MAX(CASE WHEN vital_name = 'sbp' THEN value END)
        AS sbp_max,
    AVG(CASE WHEN vital_name = 'sbp' THEN value END)
        AS sbp_mean,

    MIN(CASE WHEN vital_name = 'map' THEN value END)
        AS map_min,
    MAX(CASE WHEN vital_name = 'map' THEN value END)
        AS map_max,
    AVG(CASE WHEN vital_name = 'map' THEN value END)
        AS map_mean,

    MIN(CASE WHEN vital_name = 'temperature_c' THEN value END)
        AS temperature_c_min,
    MAX(CASE WHEN vital_name = 'temperature_c' THEN value END)
        AS temperature_c_max,
    AVG(CASE WHEN vital_name = 'temperature_c' THEN value END)
        AS temperature_c_mean

FROM cleaned_vital_events

WHERE value IS NOT NULL

GROUP BY stay_id;
"""

con.execute(vital_feature_sql)

vital_features = con.execute(
    """
    SELECT *
    FROM vital_features
    ORDER BY stay_id;
    """
).df()

print("Vital feature shape:", vital_features.shape)
vital_features.head()

Vital feature shape: (128, 19)


,stay_id,heart_rate_min,heart_rate_max,heart_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_mean,spo2_min,spo2_max,spo2_mean,sbp_min,sbp_max,sbp_mean,map_min,map_max,map_mean,temperature_c_min,temperature_c_max,temperature_c_mean
0,30057454,100.0,114.0,108.533333,13.0,25.0,18.266667,88.0,100.0,92.973684,58.0,113.0,92.611111,50.0,88.0,71.666667,36.500000,37.055556,36.698413
1,30101877,79.0,123.0,94.875000,16.0,25.0,20.440000,90.0,100.0,99.357143,102.0,166.0,137.916667,63.0,100.0,85.625000,37.388889,38.055556,37.817460
2,30425410,77.0,113.0,95.576923,14.0,29.0,18.866667,92.0,100.0,98.378378,90.0,145.0,115.360000,54.0,101.0,79.680000,36.444444,36.833333,36.674603
3,30458995,60.0,92.0,81.720000,14.0,24.0,17.600000,89.0,100.0,97.096774,104.0,157.0,129.969697,65.0,124.0,85.424242,36.333333,36.833333,36.648148
4,30585761,60.0,81.0,70.958333,10.0,24.0,19.375000,90.0,100.0,95.400000,111.0,143.0,126.958333,59.0,84.0,71.625000,36.333333,36.777778,36.638889


## 13. Inspect candidate laboratory labels

In [14]:
candidate_lab_items = con.execute(
    """
    SELECT
        itemid,
        label,
        fluid,
        category
    FROM d_labitems
    WHERE
        LOWER(label) IN (
            'creatinine',
            'urea nitrogen',
            'sodium',
            'potassium',
            'glucose',
            'white blood cells',
            'platelet count',
            'lactate',
            'albumin',
            'bicarbonate',
            'hemoglobin'
        )
        OR LOWER(label) LIKE 'bilirubin, total%'
    ORDER BY label, itemid;
    """
).df()

candidate_lab_items

,itemid,label,fluid,category
0,50862,Albumin,Blood,Chemistry
1,53085,Albumin,Blood,Chemistry
2,50882,Bicarbonate,Blood,Chemistry
3,50885,"Bilirubin, Total",Blood,Chemistry
4,53089,"Bilirubin, Total",Blood,Chemistry
5,50838,"Bilirubin, Total, Ascites",Ascites,Chemistry
6,51028,"Bilirubin, Total, Body Fluid",Other Body Fluid,Chemistry
7,51783,"Bilirubin, Total, CSF",Cerebrospinal Fluid,Chemistry
8,51812,"Bilirubin, Total, Joint Fluid",Joint Fluid,Chemistry
9,51049,"Bilirubin, Total, Pleural",Pleural,Chemistry


## 14. Map laboratory events to standardized names

Laboratory events are joined by hospital admission because `labevents` is primarily admission-level rather than ICU-stay-level.

The ICU `intime` and `prediction_time` still define the feature window.

In [15]:
lab_events_sql = """
CREATE OR REPLACE TEMP VIEW standardized_lab_events AS

SELECT
    c.stay_id,
    le.charttime,

    CASE
        WHEN LOWER(dli.label) = 'creatinine'
            THEN 'creatinine'
        WHEN LOWER(dli.label) = 'urea nitrogen'
            THEN 'bun'
        WHEN LOWER(dli.label) = 'sodium'
            THEN 'sodium'
        WHEN LOWER(dli.label) = 'potassium'
            THEN 'potassium'
        WHEN LOWER(dli.label) = 'glucose'
            THEN 'glucose'
        WHEN LOWER(dli.label) = 'white blood cells'
            THEN 'wbc'
        WHEN LOWER(dli.label) = 'platelet count'
            THEN 'platelets'
        WHEN LOWER(dli.label) = 'lactate'
            THEN 'lactate'
        WHEN LOWER(dli.label) LIKE 'bilirubin, total%'
            THEN 'bilirubin_total'
        WHEN LOWER(dli.label) = 'albumin'
            THEN 'albumin'
        WHEN LOWER(dli.label) = 'bicarbonate'
            THEN 'bicarbonate'
        WHEN LOWER(dli.label) = 'hemoglobin'
            THEN 'hemoglobin'
        ELSE NULL
    END AS lab_name,

    le.valuenum AS value

FROM base_cohort AS c

INNER JOIN labevents AS le
    ON c.hadm_id = le.hadm_id
    AND le.charttime >= c.intime
    AND le.charttime <= c.prediction_time

INNER JOIN d_labitems AS dli
    ON le.itemid = dli.itemid

WHERE
    le.valuenum IS NOT NULL
    AND (
        LOWER(dli.label) IN (
            'creatinine',
            'urea nitrogen',
            'sodium',
            'potassium',
            'glucose',
            'white blood cells',
            'platelet count',
            'lactate',
            'albumin',
            'bicarbonate',
            'hemoglobin'
        )
        OR LOWER(dli.label) LIKE 'bilirubin, total%'
    );
"""

con.execute(lab_events_sql)

lab_event_counts = con.execute(
    """
    SELECT
        lab_name,
        COUNT(*) AS measurement_count,
        COUNT(DISTINCT stay_id) AS stays_with_measurement
    FROM standardized_lab_events
    WHERE lab_name IS NOT NULL
    GROUP BY lab_name
    ORDER BY lab_name;
    """
).df()

lab_event_counts

,lab_name,measurement_count,stays_with_measurement
0,albumin,39,33
1,bicarbonate,287,127
2,bilirubin_total,85,54
3,bun,287,127
4,creatinine,286,127
5,glucose,416,127
6,hemoglobin,357,125
7,lactate,239,78
8,platelets,292,124
9,potassium,308,128


## 15. Aggregate laboratory features by ICU stay

In [16]:
lab_feature_sql = """
CREATE OR REPLACE TEMP VIEW lab_features AS

SELECT
    stay_id,

    ARG_MIN(
        CASE WHEN lab_name = 'creatinine' THEN value END,
        charttime
    ) AS creatinine_first,
    MIN(CASE WHEN lab_name = 'creatinine' THEN value END)
        AS creatinine_min,
    MAX(CASE WHEN lab_name = 'creatinine' THEN value END)
        AS creatinine_max,

    ARG_MIN(
        CASE WHEN lab_name = 'bun' THEN value END,
        charttime
    ) AS bun_first,
    MIN(CASE WHEN lab_name = 'bun' THEN value END)
        AS bun_min,
    MAX(CASE WHEN lab_name = 'bun' THEN value END)
        AS bun_max,

    ARG_MIN(
        CASE WHEN lab_name = 'sodium' THEN value END,
        charttime
    ) AS sodium_first,
    MIN(CASE WHEN lab_name = 'sodium' THEN value END)
        AS sodium_min,
    MAX(CASE WHEN lab_name = 'sodium' THEN value END)
        AS sodium_max,

    ARG_MIN(
        CASE WHEN lab_name = 'potassium' THEN value END,
        charttime
    ) AS potassium_first,
    MIN(CASE WHEN lab_name = 'potassium' THEN value END)
        AS potassium_min,
    MAX(CASE WHEN lab_name = 'potassium' THEN value END)
        AS potassium_max,

    ARG_MIN(
        CASE WHEN lab_name = 'glucose' THEN value END,
        charttime
    ) AS glucose_first,
    MIN(CASE WHEN lab_name = 'glucose' THEN value END)
        AS glucose_min,
    MAX(CASE WHEN lab_name = 'glucose' THEN value END)
        AS glucose_max,

    ARG_MIN(
        CASE WHEN lab_name = 'wbc' THEN value END,
        charttime
    ) AS wbc_first,
    MIN(CASE WHEN lab_name = 'wbc' THEN value END)
        AS wbc_min,
    MAX(CASE WHEN lab_name = 'wbc' THEN value END)
        AS wbc_max,

    ARG_MIN(
        CASE WHEN lab_name = 'platelets' THEN value END,
        charttime
    ) AS platelets_first,
    MIN(CASE WHEN lab_name = 'platelets' THEN value END)
        AS platelets_min,
    MAX(CASE WHEN lab_name = 'platelets' THEN value END)
        AS platelets_max,

    ARG_MIN(
        CASE WHEN lab_name = 'lactate' THEN value END,
        charttime
    ) AS lactate_first,
    MIN(CASE WHEN lab_name = 'lactate' THEN value END)
        AS lactate_min,
    MAX(CASE WHEN lab_name = 'lactate' THEN value END)
        AS lactate_max,

    ARG_MIN(
        CASE WHEN lab_name = 'bilirubin_total' THEN value END,
        charttime
    ) AS bilirubin_total_first,
    MIN(CASE WHEN lab_name = 'bilirubin_total' THEN value END)
        AS bilirubin_total_min,
    MAX(CASE WHEN lab_name = 'bilirubin_total' THEN value END)
        AS bilirubin_total_max,

    ARG_MIN(
        CASE WHEN lab_name = 'albumin' THEN value END,
        charttime
    ) AS albumin_first,
    MIN(CASE WHEN lab_name = 'albumin' THEN value END)
        AS albumin_min,
    MAX(CASE WHEN lab_name = 'albumin' THEN value END)
        AS albumin_max,

    ARG_MIN(
        CASE WHEN lab_name = 'bicarbonate' THEN value END,
        charttime
    ) AS bicarbonate_first,
    MIN(CASE WHEN lab_name = 'bicarbonate' THEN value END)
        AS bicarbonate_min,
    MAX(CASE WHEN lab_name = 'bicarbonate' THEN value END)
        AS bicarbonate_max,

    ARG_MIN(
        CASE WHEN lab_name = 'hemoglobin' THEN value END,
        charttime
    ) AS hemoglobin_first,
    MIN(CASE WHEN lab_name = 'hemoglobin' THEN value END)
        AS hemoglobin_min,
    MAX(CASE WHEN lab_name = 'hemoglobin' THEN value END)
        AS hemoglobin_max

FROM standardized_lab_events

WHERE lab_name IS NOT NULL

GROUP BY stay_id;
"""

con.execute(lab_feature_sql)

lab_features = con.execute(
    """
    SELECT *
    FROM lab_features
    ORDER BY stay_id;
    """
).df()

print("Lab feature shape:", lab_features.shape)
lab_features.head()

Lab feature shape: (128, 37)


,stay_id,creatinine_first,creatinine_min,creatinine_max,bun_first,bun_min,bun_max,sodium_first,sodium_min,sodium_max,potassium_first,potassium_min,potassium_max,glucose_first,glucose_min,glucose_max,wbc_first,wbc_min,wbc_max,platelets_first,platelets_min,platelets_max,lactate_first,lactate_min,lactate_max,bilirubin_total_first,bilirubin_total_min,bilirubin_total_max,albumin_first,albumin_min,albumin_max,bicarbonate_first,bicarbonate_min,bicarbonate_max,hemoglobin_first,hemoglobin_min,hemoglobin_max
0,30057454,1.8,1.7,1.8,45.0,39.0,45.0,139.0,135.0,141.0,3.3,3.3,4.2,139.0,112.0,144.0,12.7,12.7,17.8,240.0,188.0,240.0,0.6,0.6,0.6,NaN,NaN,NaN,NaN,NaN,NaN,26.0,26.0,28.0,14.4,13.1,14.4
1,30101877,1.1,0.7,1.1,26.0,18.0,26.0,135.0,135.0,137.0,4.5,4.5,4.9,153.0,153.0,216.0,19.8,19.8,22.1,438.0,438.0,468.0,NaN,NaN,NaN,0.2,0.2,0.3,3.4,3.4,3.4,22.0,22.0,25.0,10.4,9.9,10.4
2,30425410,0.5,0.5,0.5,20.0,20.0,20.0,138.0,138.0,138.0,3.9,3.9,3.9,151.0,151.0,151.0,8.5,8.5,8.5,448.0,448.0,448.0,1.5,1.5,1.5,NaN,NaN,NaN,NaN,NaN,NaN,26.0,26.0,26.0,10.0,10.0,10.0
3,30458995,0.6,0.6,0.7,11.0,10.0,11.0,134.0,134.0,140.0,3.5,3.5,4.2,136.0,100.0,136.0,12.7,12.7,21.5,247.0,214.0,247.0,1.1,1.1,1.5,0.3,0.2,0.3,NaN,NaN,NaN,23.0,23.0,26.0,11.2,11.2,12.3
4,30585761,1.3,1.3,1.3,37.0,37.0,37.0,139.0,139.0,139.0,3.8,3.8,3.8,78.0,78.0,78.0,10.5,10.5,10.5,172.0,172.0,172.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,22.0,22.0,11.0,11.0,11.0


## 16. Build the final modeling cohort

A `LEFT JOIN` keeps every patient in the base cohort, even when a vital sign or laboratory variable was not measured.

In [17]:
final_modeling_sql = """
CREATE OR REPLACE TEMP VIEW modeling_cohort AS

SELECT
    a.*,

    v.heart_rate_min,
    v.heart_rate_max,
    v.heart_rate_mean,

    v.respiratory_rate_min,
    v.respiratory_rate_max,
    v.respiratory_rate_mean,

    v.spo2_min,
    v.spo2_max,
    v.spo2_mean,

    v.sbp_min,
    v.sbp_max,
    v.sbp_mean,

    v.map_min,
    v.map_max,
    v.map_mean,

    v.temperature_c_min,
    v.temperature_c_max,
    v.temperature_c_mean,

    l.creatinine_first,
    l.creatinine_min,
    l.creatinine_max,

    l.bun_first,
    l.bun_min,
    l.bun_max,

    l.sodium_first,
    l.sodium_min,
    l.sodium_max,

    l.potassium_first,
    l.potassium_min,
    l.potassium_max,

    l.glucose_first,
    l.glucose_min,
    l.glucose_max,

    l.wbc_first,
    l.wbc_min,
    l.wbc_max,

    l.platelets_first,
    l.platelets_min,
    l.platelets_max,

    l.lactate_first,
    l.lactate_min,
    l.lactate_max,

    l.bilirubin_total_first,
    l.bilirubin_total_min,
    l.bilirubin_total_max,

    l.albumin_first,
    l.albumin_min,
    l.albumin_max,

    l.bicarbonate_first,
    l.bicarbonate_min,
    l.bicarbonate_max,

    l.hemoglobin_first,
    l.hemoglobin_min,
    l.hemoglobin_max

FROM admission_features AS a

LEFT JOIN vital_features AS v
    ON a.stay_id = v.stay_id

LEFT JOIN lab_features AS l
    ON a.stay_id = l.stay_id;
"""

con.execute(final_modeling_sql)

modeling_df = con.execute(
    """
    SELECT *
    FROM modeling_cohort
    ORDER BY stay_id;
    """
).df()

print("Final modeling dataset shape:", modeling_df.shape)
modeling_df.head()

Final modeling dataset shape: (128, 73)


,subject_id,hadm_id,stay_id,intime,prediction_time,outtime,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag,icu_admission_hour,weekend_admission,emergency_admission,transfer_admission,heart_rate_min,heart_rate_max,heart_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_mean,spo2_min,spo2_max,spo2_mean,sbp_min,sbp_max,sbp_mean,map_min,map_max,map_mean,temperature_c_min,temperature_c_max,temperature_c_mean,creatinine_first,creatinine_min,creatinine_max,bun_first,bun_min,bun_max,sodium_first,sodium_min,sodium_max,potassium_first,potassium_min,potassium_max,glucose_first,glucose_min,glucose_max,wbc_first,wbc_min,wbc_max,platelets_first,platelets_min,platelets_max,lactate_first,lactate_min,lactate_max,bilirubin_total_first,bilirubin_total_min,bilirubin_total_max,albumin_first,albumin_min,albumin_max,bicarbonate_first,bicarbonate_min,bicarbonate_max,hemoglobin_first,hemoglobin_min,hemoglobin_max
0,10023117,28872262,30057454,2171-11-14 10:06:41,2171-11-15 10:06:41,2171-11-18 20:49:43,M,53,EW EMER.,EMERGENCY ROOM,Medicare,WIDOWED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0,10,0,1,0,100.0,114.0,108.533333,13.0,25.0,18.266667,88.0,100.0,92.973684,58.0,113.0,92.611111,50.0,88.0,71.666667,36.500000,37.055556,36.698413,1.8,1.7,1.8,45.0,39.0,45.0,139.0,135.0,141.0,3.3,3.3,4.2,139.0,112.0,144.0,12.7,12.7,17.8,240.0,188.0,240.0,0.6,0.6,0.6,NaN,NaN,NaN,NaN,NaN,NaN,26.0,26.0,28.0,14.4,13.1,14.4
1,10032725,20611640,30101877,2143-03-22 06:42:00,2143-03-23 06:42:00,2143-03-25 15:05:33,F,38,EW EMER.,EMERGENCY ROOM,Other,SINGLE,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0,6,0,1,0,79.0,123.0,94.875000,16.0,25.0,20.440000,90.0,100.0,99.357143,102.0,166.0,137.916667,63.0,100.0,85.625000,37.388889,38.055556,37.817460,1.1,0.7,1.1,26.0,18.0,26.0,135.0,135.0,137.0,4.5,4.5,4.9,153.0,153.0,216.0,19.8,19.8,22.1,438.0,438.0,468.0,NaN,NaN,NaN,0.2,0.2,0.3,3.4,3.4,3.4,22.0,22.0,25.0,10.4,9.9,10.4
2,10016742,27568122,30425410,2178-07-22 08:19:00,2178-07-23 08:19:00,2178-07-25 16:42:43,F,58,OBSERVATION ADMIT,EMERGENCY ROOM,Medicaid,SINGLE,BLACK/AFRICAN AMERICAN,Medical Intensive Care Unit (MICU),0,8,0,0,0,77.0,113.0,95.576923,14.0,29.0,18.866667,92.0,100.0,98.378378,90.0,145.0,115.360000,54.0,101.0,79.680000,36.444444,36.833333,36.674603,0.5,0.5,0.5,20.0,20.0,20.0,138.0,138.0,138.0,3.9,3.9,3.9,151.0,151.0,151.0,8.5,8.5,8.5,448.0,448.0,448.0,1.5,1.5,1.5,NaN,NaN,NaN,NaN,NaN,NaN,26.0,26.0,26.0,10.0,10.0,10.0
3,10031757,28477280,30458995,2137-10-12 22:44:57,2137-10-13 22:44:57,2137-10-14 17:08:34,F,67,DIRECT EMER.,CLINIC REFERRAL,Other,DIVORCED,WHITE,Surgical Intensive Care Unit (SICU),0,22,1,1,0,60.0,92.0,81.720000,14.0,24.0,17.600000,89.0,100.0,97.096774,104.0,157.0,129.969697,65.0,124.0,85.424242,36.333333,36.833333,36.648148,0.6,0.6,0.7,11.0,10.0,11.0,134.0,134.0,140.0,3.5,3.5,4.2,136.0,100.0,136.0,12.7,12.7,21.5,247.0,214.0,247.0,1.1,1.1,1.5,0.3,0.2,0.3,NaN,NaN,NaN,23.0,23.0,26.0,11.2,11.2,12.3
4,10022281,29642388,30585761,2125-06-17 04:12:54,2125-06-18 04:12:54,2125-06-18 14:55:55,M,84,EW EMER.,EMERGENCY ROOM,Other,MARRIED,OTHER,Cardiac Vascular Intensive Care Unit (CVICU),0,4,1,1,0,60.0,81.0,70.958333,10.0,24.0,19.375000,90.0,100.0,95.400000,111.0,143.0,126.958333,59.0,84.0,71.625000,36.333333,36.777778,36.638889,1.3,1.3,1.3,37.0,37.0,37.0,139.0,139.0,139.0,3.8,3.8,3.8,78.0,78.0,78.0,10.5,10.5,10.5,172.0,172.0,172.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,22.0,22.0,11.0,11.0,11.0


## 17. Validate the final dataset

In [18]:
assert len(modeling_df) == len(base_cohort), (
    "Final dataset row count differs from the base cohort."
)

assert modeling_df["stay_id"].is_unique, (
    "Duplicate stay_id values were created."
)

assert modeling_df["hadm_id"].is_unique, (
    "Duplicate hadm_id values were created."
)

assert modeling_df["hospital_expire_flag"].isin([0, 1]).all(), (
    "The target must contain only 0 and 1."
)

assert modeling_df["intime"].notna().all()
assert modeling_df["prediction_time"].notna().all()

print("Final dataset validation passed.")

Final dataset validation passed.


## 18. Review feature missingness

In [19]:
identifier_and_time_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "intime",
    "prediction_time",
    "outtime",
]

target_column = "hospital_expire_flag"

clinical_feature_columns = [
    column
    for column in modeling_df.columns
    if column not in identifier_and_time_columns + [target_column]
]

feature_missingness = pd.DataFrame(
    {
        "feature": clinical_feature_columns,
        "missing_count": [
            modeling_df[column].isna().sum()
            for column in clinical_feature_columns
        ],
        "missing_percentage": [
            modeling_df[column].isna().mean() * 100
            for column in clinical_feature_columns
        ],
    }
).sort_values(
    "missing_percentage",
    ascending=False,
)

feature_missingness["missing_percentage"] = (
    feature_missingness["missing_percentage"].round(2)
)

feature_missingness

,feature,missing_count,missing_percentage
57,albumin_first,95,74.22
59,albumin_max,95,74.22
58,albumin_min,95,74.22
55,bilirubin_total_min,74,57.81
54,bilirubin_total_first,74,57.81
56,bilirubin_total_max,74,57.81
51,lactate_first,50,39.06
53,lactate_max,50,39.06
52,lactate_min,50,39.06
5,marital_status,10,7.81


## 19. Compare clinical feature coverage

In [20]:
coverage_summary = pd.DataFrame(
    {
        "feature_group": [
            "vital_signs",
            "laboratory_values",
        ],
        "number_of_features": [
            len([
                column for column in modeling_df.columns
                if column.startswith((
                    "heart_rate_",
                    "respiratory_rate_",
                    "spo2_",
                    "sbp_",
                    "map_",
                    "temperature_c_",
                ))
            ]),
            len([
                column for column in modeling_df.columns
                if column.startswith((
                    "creatinine_",
                    "bun_",
                    "sodium_",
                    "potassium_",
                    "glucose_",
                    "wbc_",
                    "platelets_",
                    "lactate_",
                    "bilirubin_total_",
                    "albumin_",
                    "bicarbonate_",
                    "hemoglobin_",
                ))
            ]),
        ],
    }
)

coverage_summary

,feature_group,number_of_features
0,vital_signs,18
1,laboratory_values,36


## 20. Save outputs

In [21]:
FEATURE_MISSINGNESS_PATH = (
    RESULTS_DIR / "clinical_feature_missingness.csv"
)

VITAL_CLEANING_PATH = (
    RESULTS_DIR / "vital_cleaning_summary.csv"
)

VITAL_COVERAGE_PATH = (
    RESULTS_DIR / "vital_event_coverage.csv"
)

LAB_COVERAGE_PATH = (
    RESULTS_DIR / "lab_event_coverage.csv"
)

modeling_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

feature_missingness.to_csv(
    FEATURE_MISSINGNESS_PATH,
    index=False,
)

vital_cleaning_summary.to_csv(
    VITAL_CLEANING_PATH,
    index=False,
)

vital_event_counts.to_csv(
    VITAL_COVERAGE_PATH,
    index=False,
)

lab_event_counts.to_csv(
    LAB_COVERAGE_PATH,
    index=False,
)

print("Saved:")
print("-", OUTPUT_PATH)
print("-", FEATURE_MISSINGNESS_PATH)
print("-", VITAL_CLEANING_PATH)
print("-", VITAL_COVERAGE_PATH)
print("-", LAB_COVERAGE_PATH)

Saved:
- ..\data\processed\modeling_cohort_with_clinical_features.csv
- ..\results\tables\clinical_feature_missingness.csv
- ..\results\tables\vital_cleaning_summary.csv
- ..\results\tables\vital_event_coverage.csv
- ..\results\tables\lab_event_coverage.csv


## 21. Reload and validate the saved modeling dataset

In [22]:
saved_modeling_df = pd.read_csv(
    OUTPUT_PATH,
    parse_dates=[
        "intime",
        "prediction_time",
        "outtime",
    ],
)

assert len(saved_modeling_df) == len(modeling_df)
assert saved_modeling_df["stay_id"].is_unique
assert saved_modeling_df["hadm_id"].is_unique
assert saved_modeling_df["hospital_expire_flag"].isin([0, 1]).all()

print("Saved dataset validation passed.")
print("Saved shape:", saved_modeling_df.shape)
print("Saved file:", OUTPUT_PATH.resolve())

Saved dataset validation passed.
Saved shape: (128, 73)
Saved file: C:\Projects\clinical-outcome-prediction\data\processed\modeling_cohort_with_clinical_features.csv


## Notebook 4 Summary

This notebook:

- connected Python to DuckDB;
- registered MIMIC-IV compressed CSV files as SQL views;
- loaded the base ICU cohort;
- engineered admission-hour, weekend, emergency, and transfer features;
- extracted vital signs from the first 24 hours after ICU admission;
- applied broad plausibility checks;
- extracted laboratory measurements from the same 24-hour window;
- aggregated measurements into patient-level features;
- joined all features without dropping patients;
- validated and saved the final dataset.

## Main output

```text
data/processed/modeling_cohort_with_clinical_features.csv
```

## Next step

Notebook 5 should:

1. examine missingness and distributions in the expanded dataset;
2. define numerical and categorical predictors;
3. create a scikit-learn preprocessing pipeline;
4. fit imputation and encoding on training data only.